This routine generates **Figure 4** and its supplementary panels for the paper *"Global Potential of Potable Reuse Across Coupled Climate and Socioeconomic Futures"*. If you have any questions please contact [a.sarfraz@uu.nl](mailto:a.sarfraz@uu.nl).


### Imports

In [1]:
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error

import shap
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

### Setting directories

In [2]:
from __future__ import annotations

import sys
import yaml

# Resolve the repo root from this notebook's location.
_NB_DIR   = Path.cwd()
REPO_ROOT = _NB_DIR.parent if _NB_DIR.name == "notebooks" else _NB_DIR

# Expose the shared package.
sys.path.insert(0, str(REPO_ROOT / "src"))

# Load the path config.
_cfg = yaml.safe_load((REPO_ROOT / "config" / "paths.yaml").read_text())

def _resolve(p):
    p = Path(p)
    return p if p.is_absolute() else (REPO_ROOT / p).resolve()

PATHS = {
    "data":    {k: _resolve(v) for k, v in _cfg["data"].items()},
    "outputs": {k: _resolve(v) for k, v in _cfg["outputs"].items()},
}

from potable_reuse.style import apply_style, AXIS_LABEL_SIZE, SAVE_DPI
apply_style()

MERGED_DIR = PATHS["data"]["merged_parquets_dir"]
FIG_DIR    = PATHS["outputs"]["figure4"]


(FIG_DIR / "figures").mkdir(parents=True, exist_ok=True)
(FIG_DIR / "supplementary").mkdir(parents=True, exist_ok=True)
(FIG_DIR / "tables").mkdir(parents=True, exist_ok=True)
(FIG_DIR / "models").mkdir(parents=True, exist_ok=True)

TUNE         = True   
RANDOM_STATE = 42

print('Repo root  :', REPO_ROOT)
print('Merged dir :', MERGED_DIR)
print('Tune GBT   :', TUNE)

Repo root  : C:\Users\Sarfr001\Documents\Global_Potential_Potable_Reuse
Merged dir : C:\Users\Sarfr001\Documents\Global_Potential_Potable_Reuse\data\merged_parquets
Tune GBT   : True


### Constants

Lagged design: features at 2050 under PR0, target at 2100 under PR100. 

In [3]:

HEADLINE = dict(
    feat_year   = 2050,
    target_year = 2100,
    pr_level    = 'PR100',
)
TARGET    = 'pct_reduction'

# Exemplar regions 
EXEMPLARS = ['Middle East', 'Pakistan', 'USA', 'Japan']

# renaming 
FEATURE_PRETTY = {
    'mun_w_pr0':    'Municipal withdrawal',
    'pop':          'Population',
    'gdp_pcap_mer': 'GDP per capita',
    'gw_w_pr0':     'Groundwater withdrawal',
}
REGIONAL_FEATURES_BASE = ['mun_w_pr0', 'pop', 'gdp_pcap_mer', 'gw_w_pr0']

# Reuse-cost tiers.
RC_MAP   = {'LRC': 0, 'MRC': 1, 'HRC': 2}
RC_ORDER = ['LRC', 'MRC', 'HRC']

# Tier colors used in the cross-tier SI panel.
COL_LRC, COL_MRC, COL_HRC = '#1f4e99', '#e8923c', '#b03020'

EXEMPLAR_COLORS = {
    'USA':         '#1f77b4',
    'Middle East': '#a8324a',
    'Pakistan':    '#9467bd',
    'Japan':       '#1f77b4',
}


REGION_ABBREV = {
    'Middle East': 'M. East',
    'Pakistan':    'Pak',
    'USA':         'USA',
    'Japan':       'Japan',
}


def feature_columns(feat_year):
    """Predictor column names with the feature-year suffix attached."""
    return [f'{c}_{feat_year}' for c in REGIONAL_FEATURES_BASE]


def pretty_name(col):
    """Map a raw predictor column name to its display label."""
    if col in FEATURE_PRETTY:
        return FEATURE_PRETTY[col]
    parts = col.rsplit('_', 1)
    if len(parts) == 2 and parts[1].isdigit() and parts[0] in FEATURE_PRETTY:
        return FEATURE_PRETTY[parts[0]]
    return col

### Data 

Reads the merged parquets and builds the per-region design matrix. The target is the percentage reduction in regional municipal withdrawal at the target year, comparing the chosen PR level to its matched PR0 baseline. Features are the four regional attributes at the feature year under PR0.

In [4]:
def _filter(df, year_filter=None, pr_filter=None):
    """Restrict a parquet to the requested years and PR levels."""
    if year_filter is not None and 'year' in df.columns:
        if isinstance(year_filter, (int, float)):
            year_filter = [float(year_filter)]
        df = df[df['year'].isin([float(y) for y in year_filter])]
    if pr_filter is not None and 'pr_level' in df.columns:
        if isinstance(pr_filter, str):
            pr_filter = [pr_filter]
        df = df[df['pr_level'].isin(pr_filter)]
    return df


def load_merged(merged_dir, feat_year, target_year, pr_level):
    """Read every parquet needed for the design matrix and pre-filter."""
    md = Path(merged_dir)
    print(f'[load] reading merged parquets from {md}')
    muni = pd.read_parquet(md / 'water_td_muni_W.parquet')
    pop  = pd.read_parquet(md / 'population_by_region.parquet')
    gdp  = pd.read_parquet(md / 'gdp_pcap_mer_by_region.parquet')
    rg   = pd.read_parquet(md / 'water_withdrawals_runoff_vs_groundwater.parquet')

    years_muni = sorted({feat_year, target_year})
    return {
        'muni':      _filter(muni, year_filter=years_muni,
                             pr_filter=['PR0', pr_level]),
        'pop':       _filter(pop,  year_filter=feat_year, pr_filter='PR0'),
        'gdp':       _filter(gdp,  year_filter=feat_year, pr_filter='PR0'),
        'runoff_gw': _filter(rg,   year_filter=feat_year, pr_filter='PR0'),
    }


def build_target(muni_all, target_year, pr_level):
    """Per-(scenario, region) percentage-reduction target at target_year.

    """
    keys_full = ['ssp', 'rcp', 'reuse_cost', 'supply_capacity', 'region']
    keys_pr0  = ['ssp', 'rcp',               'supply_capacity', 'region']

    snap = muni_all[muni_all['year'] == target_year]
    pr0 = (snap[snap['pr_level'] == 'PR0']
           .groupby(keys_pr0, as_index=False)['value'].mean()
           .rename(columns={'value': 'mun_w_pr0_target'}))
    prN = (snap[snap['pr_level'] == pr_level]
           .groupby(keys_full, as_index=False)['value'].sum()
           .rename(columns={'value': f'mun_w_{pr_level.lower()}'}))

    out = prN.merge(pr0, on=keys_pr0, how='inner')
    out['abs_reduction'] = (out['mun_w_pr0_target']
                            - out[f'mun_w_{pr_level.lower()}'])
    out['pct_reduction'] = (100.0 * out['abs_reduction']
                            / out['mun_w_pr0_target'].replace(0, np.nan))
    out = out.dropna(subset=['pct_reduction'])
    print(f'[target] {len(out):,} rows  '
          f'mean pct={out["pct_reduction"].mean():.1f}%')
    return out


def build_features(loaded, feat_year):
    """Four-feature regional design matrix at feat_year under PR0."""
    keys = ['ssp', 'rcp', 'supply_capacity', 'region']

    def _pr0_sum(df, newname):
        s = df[(df['year'] == feat_year) & (df['pr_level'] == 'PR0')]
        return (s.groupby(keys, as_index=False)['value'].sum()
                 .rename(columns={'value': newname}))

    def _pr0_mean(df, newname):
        s = df[(df['year'] == feat_year) & (df['pr_level'] == 'PR0')]
        return (s.groupby(keys, as_index=False)['value'].mean()
                 .rename(columns={'value': newname}))

    mun = _pr0_sum (loaded['muni'], f'mun_w_pr0_{feat_year}')
    pop = _pr0_mean(loaded['pop'],  f'pop_{feat_year}')
    gdp = _pr0_mean(loaded['gdp'],  f'gdp_pcap_mer_{feat_year}')

    # Groundwater is the subset of the runoff_vs_groundwater parquet
    # whose `subresource` field starts with 'groundwater'.
    rg = loaded['runoff_gw']
    rg = rg[(rg['year'] == feat_year) & (rg['pr_level'] == 'PR0')].copy()
    rg['is_gw'] = rg['subresource'].astype(str).str.lower().str.startswith('groundwater')
    gw = (rg[rg['is_gw']].groupby(keys, as_index=False)['value'].sum()
          .rename(columns={'value': f'gw_w_pr0_{feat_year}'}))

    feat = mun
    for part in [pop, gdp, gw]:
        feat = feat.merge(part, on=keys, how='outer')
    print(f'[features] {len(feat):,} rows  cols={list(feat.columns)}')
    return feat


def assemble_matrix(target, features):
    """Merge target onto features and add the reuse-cost ordinal."""
    keys = ['ssp', 'rcp', 'supply_capacity', 'region']
    df = target.merge(features, on=keys, how='left')
    df['rc_ord'] = df['reuse_cost'].map(RC_MAP)
    return df

### Build the design matrix

In [5]:
fy = HEADLINE['feat_year']
ty = HEADLINE['target_year']
pr = HEADLINE['pr_level']

loaded   = load_merged(MERGED_DIR, feat_year=fy, target_year=ty, pr_level=pr)
target   = build_target(loaded['muni'], target_year=ty, pr_level=pr)
features = build_features(loaded, fy)
matrix   = assemble_matrix(target, features)

feat_cols = feature_columns(fy)
matrix    = matrix.dropna(subset=feat_cols + [TARGET]).reset_index(drop=True)

print(f'\n[matrix] {len(matrix):,} rows x {len(feat_cols)} features')
print('[matrix] features =', feat_cols)
print(f'[matrix] mean {TARGET} = {matrix[TARGET].mean():.1f}%')
matrix.head(3)

[load] reading merged parquets from C:\Users\Sarfr001\Documents\Global_Potential_Potable_Reuse\data\merged_parquets
[target] 4,896 rows  mean pct=28.4%
[features] 1,632 rows  cols=['ssp', 'rcp', 'supply_capacity', 'region', 'mun_w_pr0_2050', 'pop_2050', 'gdp_pcap_mer_2050', 'gw_w_pr0_2050']

[matrix] 4,896 rows x 4 features
[matrix] features = ['mun_w_pr0_2050', 'pop_2050', 'gdp_pcap_mer_2050', 'gw_w_pr0_2050']
[matrix] mean pct_reduction = 28.4%


,ssp,rcp,reuse_cost,supply_capacity,region,mun_w_pr100,mun_w_pr0_target,abs_reduction,pct_reduction,mun_w_pr0_2050,pop_2050,gdp_pcap_mer_2050,gw_w_pr0_2050,rc_ord
0,SSP1,2p6,HRC,H,Africa_Eastern,15.35310,15.5385,0.18540,1.193165,29.61558,558904.0,2.02169,68.9925,2
1,SSP1,2p6,LRC,H,Africa_Eastern,8.70771,15.5385,6.83079,43.960421,29.61558,558904.0,2.02169,68.9925,0
2,SSP1,2p6,MRC,H,Africa_Eastern,11.82590,15.5385,3.71260,23.892911,29.61558,558904.0,2.02169,68.9925,1


### Per-tier model fitting

CART (depth 3), GBT for SHAP attribution, RF kept as a robustness check in the supplementary metrics table.

In [6]:
def _tune_gbt(X_tr, y_tr, random_state):
    """5-fold CV grid search over a small GBT hyperparameter grid."""
    grid = {
        'n_estimators':     [200, 500],
        'max_depth':        [3, 5],
        'learning_rate':    [0.05, 0.1],
        'min_samples_leaf': [10, 20],
    }
    gs = GridSearchCV(
        GradientBoostingRegressor(random_state=random_state),
        grid, cv=5, scoring='r2', n_jobs=-1,
    )
    gs.fit(X_tr, y_tr)
    return gs.best_estimator_, gs.best_params_


def fit_tier(matrix, feat_cols, rc_name, tune=True,
             random_state=RANDOM_STATE, verbose=True):
    """Fit CART, RF, and GBT for one reuse-cost tier."""
    if verbose:
        print(f'\n[fit] {rc_name}')
    sub = matrix[matrix['rc_ord'] == RC_MAP[rc_name]].reset_index(drop=True)
    X = sub[feat_cols].astype(float)
    y = sub[TARGET].astype(float)
    regions = sub['region']

    X_tr, X_te, y_tr, y_te, r_tr, r_te = train_test_split(
        X, y, regions, test_size=0.20, random_state=random_state)

    cart = DecisionTreeRegressor(
        max_depth=3, min_samples_leaf=20,
        random_state=random_state,
    ).fit(X_tr, y_tr)

    rf = RandomForestRegressor(
        n_estimators=500, min_samples_leaf=20,
        n_jobs=-1, random_state=random_state,
    ).fit(X_tr, y_tr)

    if tune:
        gbt, gbt_params = _tune_gbt(X_tr, y_tr, random_state)
    else:
        gbt = GradientBoostingRegressor(
            n_estimators=500, max_depth=3,
            learning_rate=0.1, min_samples_leaf=20,
            random_state=random_state,
        ).fit(X_tr, y_tr)
        gbt_params = {
            'n_estimators': 500, 'max_depth': 3,
            'learning_rate': 0.1, 'min_samples_leaf': 20,
        }

    metrics = {'n_total': len(sub)}
    for m, name in [(cart, 'cart'), (rf, 'rf'), (gbt, 'gbt')]:
        metrics[name] = {
            'train_r2': r2_score(y_tr, m.predict(X_tr)),
            'test_r2':  r2_score(y_te, m.predict(X_te)),
            'test_mae': mean_absolute_error(y_te, m.predict(X_te)),
        }
    metrics['gbt']['best_params'] = gbt_params

    if verbose:
        for m in ('cart', 'rf', 'gbt'):
            print(f'  {m.upper():<4} '
                  f'train R2={metrics[m]["train_r2"]:.3f}  '
                  f'test R2={metrics[m]["test_r2"]:.3f}')

    return {
        'rc_name':  rc_name, 'feat_cols': feat_cols,
        'X_train':  X_tr, 'X_test':  X_te,
        'y_train':  y_tr, 'y_test':  y_te,
        'r_train':  r_tr, 'r_test':  r_te,
        'cart': cart, 'rf': rf, 'gbt': gbt,
        'metrics': metrics,
    }


fits = {tier: fit_tier(matrix, feat_cols, tier, tune=TUNE)
        for tier in RC_ORDER}


[fit] LRC
  CART train R2=0.484  test R2=0.468
  RF   train R2=0.726  test R2=0.700
  GBT  train R2=0.998  test R2=0.996

[fit] MRC
  CART train R2=0.618  test R2=0.566
  RF   train R2=0.883  test R2=0.874
  GBT  train R2=1.000  test R2=0.999

[fit] HRC
  CART train R2=0.413  test R2=0.334
  RF   train R2=0.802  test R2=0.764
  GBT  train R2=1.000  test R2=0.999


### SHAP per RC tier

In [7]:
def compute_shap(fit, max_samples=1500, random_state=RANDOM_STATE):
    """TreeExplainer SHAP values on a sample of the training rows."""
    X_tr = fit['X_train']
    n = min(max_samples, len(X_tr))
    X_shap = X_tr.sample(n, random_state=random_state)
    sv = shap.TreeExplainer(fit['gbt']).shap_values(X_shap)
    return {
        'shap_values':   sv,
        'X_shap':        X_shap,
        'regions_shap':  fit['r_train'].loc[X_shap.index],
    }


def shap_per_feature(shap_pkg, feat_cols):
    """Per-feature mean |SHAP| and mean signed SHAP, sorted descending."""
    sv = shap_pkg['shap_values']
    return (pd.DataFrame({
        'feature':     feat_cols,
        'pretty':      [pretty_name(c) for c in feat_cols],
        'mean_abs':    np.abs(sv).mean(axis=0),
        'mean_signed': sv.mean(axis=0),
    })
    .sort_values('mean_abs', ascending=False)
    .reset_index(drop=True))


shaps = {tier: compute_shap(fits[tier]) for tier in RC_ORDER}

for tier in RC_ORDER:
    df = shap_per_feature(shaps[tier], feat_cols)
    print(f'\n=== {tier} feature ranking ===')
    print(df[['pretty', 'mean_abs', 'mean_signed']]
          .round(3).to_string(index=False))


=== LRC feature ranking ===
                pretty  mean_abs  mean_signed
        GDP per capita     2.964       -0.431
Groundwater withdrawal     2.072        0.240
            Population     1.593        0.131
  Municipal withdrawal     1.573        0.060

=== MRC feature ranking ===
                pretty  mean_abs  mean_signed
Groundwater withdrawal     6.288        0.165
            Population     3.221        0.154
        GDP per capita     2.737       -0.369
  Municipal withdrawal     2.697        0.049

=== HRC feature ranking ===
                pretty  mean_abs  mean_signed
Groundwater withdrawal     1.184       -0.116
        GDP per capita     0.845       -0.014
            Population     0.593        0.076
  Municipal withdrawal     0.580        0.054


In [8]:
def leaf_region_assignments(
    fit, matrix, feat_cols, exemplars=EXEMPLARS,
    tier='MRC', share_threshold=0.30, rank_top=3,
):
    """Return ({leaf_id: [regions]}, full_counts_dataframe)."""
    cart = fit['cart']
    X_tr = fit['X_train']
    leaf_ids = cart.apply(X_tr)

    sub = matrix[matrix['rc_ord'] == RC_MAP[tier]].reset_index(drop=True)
    region_train = sub.loc[X_tr.index, 'region'].values

    df = pd.DataFrame({'leaf': leaf_ids, 'region': region_train})
    counts = (df.groupby(['leaf', 'region']).size()
                .rename('count').reset_index())
    region_totals = df.groupby('region').size().rename('region_total')
    counts = counts.merge(region_totals, on='region', how='left')
    counts['share_of_region'] = counts['count'] / counts['region_total']
    counts['leaf_rank'] = (counts.groupby('leaf')['count']
                           .rank(method='first', ascending=False).astype(int))
    counts['exemplar'] = counts['region'].isin(exemplars)
    counts['chip']    = (counts['exemplar']
                         & (counts['share_of_region'] >= share_threshold)
                         & (counts['leaf_rank'] <= rank_top))

    chips = {}
    for leaf in sorted(counts['leaf'].unique()):
        these = counts[(counts['leaf'] == leaf) & counts['chip']]
        chips[int(leaf)] = (these.sort_values('count', ascending=False)
                                  ['region'].tolist())
    return chips, counts

### Save helper

In [9]:
#3 feel free to comment out any format
def save_fig(fig, out, stem):
    """Save figure as SVG, PNG, and a self-contained HTML wrapper.
    """
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    for ext in ('svg', 'png'):
        fig.savefig(out / f'{stem}.{ext}', bbox_inches='tight')
    svg = (out / f'{stem}.svg').read_text(encoding='utf-8')
    (out / f'{stem}.html').write_text(
        "<!doctype html><meta charset='utf-8'>"
        f"<title>{stem}</title>"
        "<body style='font-family:Arial,sans-serif;"
        "margin:24px;background:#fafafa'>"
        f"<h3>{stem}</h3>{svg}</body>",
        encoding='utf-8',
    )
    print(f'  Saved: {stem}.svg | {stem}.png | {stem}.html')
    plt.close(fig)

### Custom CART 

In [10]:

_LEAF_CMAP = cm.get_cmap('Greens')
_LEAF_NBINS = 5


def _value_to_fill(v, vmin, vmax, n_bins=_LEAF_NBINS):
    """Map a leaf's predicted reduction onto a discrete green band.

    """
    if vmax <= vmin:
        return _LEAF_CMAP(0.5)[:3]
    width = (vmax - vmin) / n_bins
    idx = int(np.clip(np.floor((v - vmin) / width), 0, n_bins - 1))
    t = (idx + 0.5) / n_bins
    return _LEAF_CMAP(0.20 + 0.55 * t)[:3]


def _format_threshold(feature_pretty, threshold, depth=0):
    """Threshold label for an internal node.

    """
    is_root = (depth == 0)
    is_deep = (depth >= 2)

    if 'Groundwater' in feature_pretty:
        # Groundwater withdrawal is in km^3 yr^-1.
        if is_root:
            return f'Groundwater\n\u2264 {threshold:.0f} km\u00b3 yr\u207b\u00b9'
        return f'Groundwater\n\u2264 {threshold:.0f}'

    if 'GDP' in feature_pretty:
        label = 'GDP per cap.' if is_deep else 'GDP per capita'
        return f'{label}\n\u2264 \\${threshold:.1f}k'

    if 'Population' in feature_pretty:
        # GCAM population is in thousands; convert to millions for legibility.
        if threshold >= 1e3:
            return f'Population\n\u2264 {threshold/1e3:.0f} M'
        return f'Population\n\u2264 {threshold:.0f} k'

    if 'Municipal' in feature_pretty:
        label = 'Mun. with.' if is_deep else 'Mun. withdrawal'
        return f'{label}\n\u2264 {threshold:.0f}'

    return f'{feature_pretty}\n\u2264 {threshold:.2f}'


def _build_layout(tree):
    """Compute (x, y, depth, is_leaf, parent) arrays for every node.

    """
    n = tree.node_count
    children_l = tree.children_left
    children_r = tree.children_right

    depth = np.zeros(n, dtype=int)
    parent = -np.ones(n, dtype=int)
    stack = [(0, 0)]
    while stack:
        node, d = stack.pop()
        depth[node] = d
        if children_l[node] != -1:
            parent[children_l[node]] = node
            parent[children_r[node]] = node
            stack.append((children_l[node], d + 1))
            stack.append((children_r[node], d + 1))
    is_leaf = (children_l == -1)
    max_depth = depth.max()


    leaf_order = []
    def dfs(node):
        if children_l[node] == -1:
            leaf_order.append(node)
            return
        dfs(children_l[node])
        dfs(children_r[node])
    dfs(0)

    x_pos = np.zeros(n)
    n_leaves = len(leaf_order)
    if n_leaves == 1:
        x_pos[leaf_order[0]] = 0.5
    else:
        for i, node in enumerate(leaf_order):
            x_pos[node] = 0.05 + (0.90 * i) / (n_leaves - 1)

    def fill_internal(node):
        if children_l[node] == -1:
            return x_pos[node]
        xl = fill_internal(children_l[node])
        xr = fill_internal(children_r[node])
        x_pos[node] = 0.5 * (xl + xr)
        return x_pos[node]
    fill_internal(0)

    y_top, y_bot = 0.92, 0.22
    if max_depth == 0:
        y_pos = np.full(n, 0.5)
    else:
        y_pos = y_top - (depth / max_depth) * (y_top - y_bot)
    return x_pos, y_pos, depth, is_leaf, parent


def _chip_label_and_color(regions):
    """Combine 1+ regions into one chip label + a single fill color.
    """
    if len(regions) == 1:
        return regions[0], EXEMPLAR_COLORS[regions[0]]
    abbreviated = [REGION_ABBREV.get(r, r) for r in regions[:2]]
    label = ' \u00b7 '.join(abbreviated)
    return label, EXEMPLAR_COLORS[regions[0]]


def draw_custom_cart(
    ax, fit, feat_cols, chips_for_leaf,
    panel_label='a',
    panel_subtitle='Regression tree, medium reuse cost',
    show_panel_label=True, show_colorbar=True,
):
    tree = fit['cart'].tree_
    pretty_feats = [pretty_name(c) for c in feat_cols]
    x_pos, y_pos, depth, is_leaf, parent = _build_layout(tree)

    leaf_values = tree.value[is_leaf, 0, 0]
    vmin = float(np.min(leaf_values))
    vmax = float(np.max(leaf_values))
    n_total = float(tree.n_node_samples[0])

    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')


    for node in range(tree.node_count):
        if parent[node] == -1:
            continue
        s_internal = not is_leaf[parent[node]]
        d_internal = not is_leaf[node]
        sy_top = y_pos[parent[node]] - (0.075 if s_internal else 0.04)
        dy_bot = y_pos[node]         + (0.075 if d_internal else 0.04)
        ax.plot([x_pos[parent[node]], x_pos[node]], [sy_top, dy_bot],
                color='#888', linewidth=0.7, zorder=1)

        if depth[node] == 1 and not is_leaf[parent[node]]:
            mid_x = 0.5 * (x_pos[parent[node]] + x_pos[node])
            mid_y = 0.5 * (sy_top + dy_bot)
            text = 'yes' if x_pos[node] < x_pos[parent[node]] else 'no'
            ax.text(mid_x, mid_y, text, ha='center', va='center',
                    fontsize=7.5, color='#444', style='italic',
                    zorder=2,
                    bbox=dict(facecolor='white', edgecolor='none',
                              pad=0.5))

    # Nodes.
    for node in range(tree.node_count):
        x, y = x_pos[node], y_pos[node]
        n_pct = 100.0 * tree.n_node_samples[node] / n_total
        mu    = float(tree.value[node, 0, 0])
        d     = int(depth[node])

        if not is_leaf[node]:
            feat_idx = int(tree.feature[node])
            thresh   = float(tree.threshold[node])
            label    = _format_threshold(pretty_feats[feat_idx], thresh, depth=d)
            w, h     = 0.155, 0.135
            box = FancyBboxPatch(
                (x - w/2, y - h/2), w, h,
                boxstyle="round,pad=0.005,rounding_size=0.013",
                linewidth=0.8, edgecolor='#333',
                facecolor='#f4efe6', zorder=3,
            )
            ax.add_patch(box)
            ax.text(x, y, label, ha='center', va='center',
                    fontsize=8, linespacing=1.15, fontweight='bold',
                    zorder=4)
        else:
            chips_here = chips_for_leaf.get(int(node), [])
            color = _value_to_fill(mu, vmin, vmax)
            w, h = 0.085, 0.075

            # Highlight border for leaves that qualify for a chip.
            if chips_here:
                _, chip_color = _chip_label_and_color(chips_here)
                edgecolor = chip_color
                linewidth = 1.6
            else:
                edgecolor = '#333'
                linewidth = 0.8

            box = FancyBboxPatch(
                (x - w/2, y - h/2), w, h,
                boxstyle="round,pad=0.003,rounding_size=0.012",
                linewidth=linewidth, edgecolor=edgecolor,
                facecolor=color, zorder=3,
            )
            ax.add_patch(box)
            ax.text(x, y + 0.013, f'\u03bc = {mu:.0f}%',
                    ha='center', va='center', fontsize=8,
                    fontweight='bold', color='#111', zorder=4)
            ax.text(x, y - 0.013, f'n = {n_pct:.0f}%',
                    ha='center', va='center', fontsize=7, color='#222',
                    zorder=4)

            if chips_here:
                label, fill = _chip_label_and_color(chips_here)
                chip_w = 0.110 if len(chips_here) > 1 else 0.080
                chip_h = 0.027
                cy = y + h/2 + 0.020
                chip_box = FancyBboxPatch(
                    (x - chip_w/2, cy - chip_h/2), chip_w, chip_h,
                    boxstyle="round,pad=0.004,rounding_size=0.013",
                    facecolor=fill, edgecolor='white', linewidth=0.8,
                    zorder=5,
                )
                ax.add_patch(chip_box)
                ax.text(x, cy, label, ha='center', va='center',
                        fontsize=7, color='white', fontweight='bold',
                        zorder=6)

    # Discrete center-bottom legend for the leaf-fill bands. 
    if show_colorbar:
        from matplotlib.colors import BoundaryNorm, ListedColormap
        import matplotlib as _mpl

        n_bins = 5
        levels = np.linspace(vmin, vmax, n_bins + 1)
        band_colors = [_value_to_fill(0.5 * (levels[i] + levels[i + 1]),
                                       vmin, vmax)
                       for i in range(n_bins)]
        discrete_cmap = ListedColormap(band_colors)
        norm = BoundaryNorm(levels, ncolors=n_bins)

        cb_w, cb_h = 0.30, 0.030
        cax = ax.inset_axes([0.5 - cb_w / 2, 0.05, cb_w, cb_h])
        sm = _mpl.cm.ScalarMappable(cmap=discrete_cmap, norm=norm)
        sm.set_array([])
        cb = plt.colorbar(
            sm, cax=cax, orientation='horizontal',
            ticks=levels, spacing='proportional',
        )
        cb.ax.set_xticklabels([f'{l:.0f}%' for l in levels],
                              fontsize=16)
        cb.outline.set_linewidth(0.4)
        cb.ax.tick_params(length=2)
        cb.set_label(
            'Mean predicted reduction (\u03bc)',
            fontsize=16, color='#444', labelpad=4,
        )

    if show_panel_label:
        ax.text(0.02, 0.98, panel_label, fontsize=16, fontweight='bold',
                ha='left', va='top', transform=ax.transAxes)
        ax.text(0.06, 0.98, panel_subtitle,
                fontsize=16, color='#444',
                ha='left', va='top', transform=ax.transAxes)

### SHAP beeswarm 

In [11]:
_SHAP_CMAP = LinearSegmentedColormap.from_list(
    'shap_blue_red',
    ['#3b8df0', '#8a6bd6', '#e0408c', '#d63a5c'], N=256,
)


def _draw_shap_beeswarm(
    ax, shap_pkg, feat_cols,
    panel_label='b',
    panel_subtitle='SHAP attribution, medium reuse cost',
    show_panel_label=True, jitter=0.20, point_size=12,
):
    """Custom beeswarm-style scatter with one row per feature.

    """
    sv = shap_pkg['shap_values']
    X  = shap_pkg['X_shap']
    pretty_feats = [pretty_name(c) for c in feat_cols]

    order_idx = np.argsort(-np.abs(sv).mean(axis=0))
    n_feats = len(feat_cols)
    y_positions = np.arange(n_feats)[::-1]   # top row = highest |SHAP|
    rng = np.random.default_rng(RANDOM_STATE)

    for plot_row, j in enumerate(order_idx):
        s = sv[:, j]
        x_vals = X.iloc[:, j].values.astype(float)
        if np.nanmax(x_vals) > np.nanmin(x_vals):
            v_norm = ((x_vals - np.nanmin(x_vals))
                      / (np.nanmax(x_vals) - np.nanmin(x_vals)))
        else:
            v_norm = np.full_like(x_vals, 0.5)
        y_jitter = rng.uniform(-jitter, jitter, size=len(s))
        y_row = np.full_like(s, y_positions[plot_row], dtype=float) + y_jitter
        ax.scatter(s, y_row, c=v_norm, cmap=_SHAP_CMAP,
                   s=point_size, alpha=0.65, edgecolor='none',
                   vmin=0, vmax=1)

    ax.axvline(0, color='#888', linewidth=0.6, zorder=0,
               linestyle='--')
    ax.set_yticks(y_positions)
    ax.set_yticklabels([pretty_feats[j] for j in order_idx], fontsize=16)
    ax.set_xlabel('SHAP value (impact on predicted % reduction)',
                  fontsize=16)
    ax.tick_params(axis='x', labelsize=14)

    # Inset feature-value colorbar with 16-pt labels to match panel (a).
    cax = ax.inset_axes([1.02, 0.10, 0.025, 0.80])
    sm = plt.cm.ScalarMappable(cmap=_SHAP_CMAP, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cb = plt.colorbar(sm, cax=cax)
    cb.set_ticks([0.05, 0.95])
    cb.set_ticklabels(['Low', 'High'])
    cb.ax.tick_params(labelsize=16)
    cb.set_label('Attribute value', fontsize=16, rotation=270, labelpad=20)
    cb.outline.set_linewidth(0.4)

    if show_panel_label:
        ax.text(-0.18, 1.02, panel_label, fontsize=16, fontweight='bold',
                ha='left', va='bottom', transform=ax.transAxes)
        ax.text(-0.13, 1.02, panel_subtitle, fontsize=16, color='#444',
                ha='left', va='bottom', transform=ax.transAxes)

## Generate the final two-panel Figure 4

Vertical layout: panel (a) regression tree on top, panel (b) SHAP beeswarm below. Both panels are at the medium reuse-cost tier.

In [18]:
TIER = 'MRC'  # headline tier

# MRC is the headline Figure 4; LRC and HRC are the
# supplementary equivalents the manuscript 
RC_WORD = {'LRC': 'low', 'MRC': 'medium', 'HRC': 'high'}
TIER_OUTPUT = {
    'MRC': (FIG_DIR / 'figures',       'figure4_main'),
    'LRC': (FIG_DIR / 'supplementary', 'figSx_cart_shap_LRC'),
    'HRC': (FIG_DIR / 'supplementary', 'figSx_cart_shap_HRC'),
}

for tier in RC_ORDER:
    fit_t = fits[tier]

    # Leaf-to-region chips for this tier, plus the per-leaf roster table.
    chips_t, leaf_table = leaf_region_assignments(
        fit_t, matrix, feat_cols, tier=tier,
    )
    leaf_table.to_csv(
        FIG_DIR / 'tables' / f'cart_leaf_regions_{tier}.csv', index=False,
    )

    cost_word     = RC_WORD[tier]
    out_dir, stem = TIER_OUTPUT[tier]

    fig = plt.figure(figsize=(14, 13.0))
    gs = GridSpec(
        2, 1, height_ratios=[1.10, 1.10],
        hspace=0.22,
        left=0.10, right=0.94, top=0.97, bottom=0.06,
    )
    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[1, 0])

    draw_custom_cart(
        ax_a, fit_t, feat_cols, chips_t,
        panel_label='a',
        panel_subtitle=f'Regression tree, {cost_word} reuse cost',
    )
    _draw_shap_beeswarm(
        ax_b, shaps[tier], feat_cols,
        panel_label='b',
        panel_subtitle=f'SHAP attribution, {cost_word} reuse cost',
    )

    save_fig(fig, out_dir, stem)
    print(f'Wrote {tier} two-panel figure to:',
          (out_dir / f'{stem}.svg').resolve())

  Saved: figSx_cart_shap_LRC.svg | figSx_cart_shap_LRC.png | figSx_cart_shap_LRC.html
Wrote LRC two-panel figure to: C:\Users\Sarfr001\Documents\Global_Potential_Potable_Reuse\outputs\figure4\supplementary\figSx_cart_shap_LRC.svg
  Saved: figure4_main.svg | figure4_main.png | figure4_main.html
Wrote MRC two-panel figure to: C:\Users\Sarfr001\Documents\Global_Potential_Potable_Reuse\outputs\figure4\figures\figure4_main.svg
  Saved: figSx_cart_shap_HRC.svg | figSx_cart_shap_HRC.png | figSx_cart_shap_HRC.html
Wrote HRC two-panel figure to: C:\Users\Sarfr001\Documents\Global_Potential_Potable_Reuse\outputs\figure4\supplementary\figSx_cart_shap_HRC.svg


## Supplementary panels


### Standalone SHAP beeswarms per tier

One beeswarm per tier in the same idiom as panel (b).

In [13]:
def supp_shap_tier(shap_pkg, feat_cols, out_dir, tier):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    _draw_shap_beeswarm(
        ax, shap_pkg, feat_cols,
        panel_label='', show_panel_label=False,
        panel_subtitle=f'SHAP attribution, {tier}',
    )
    save_fig(fig, out_dir, f'figS_shap_{tier}')


for tier in RC_ORDER:
    supp_shap_tier(shaps[tier], feat_cols,
                   FIG_DIR / 'supplementary', tier)

  Saved: figS_shap_LRC.svg | figS_shap_LRC.png | figS_shap_LRC.html
  Saved: figS_shap_MRC.svg | figS_shap_MRC.png | figS_shap_MRC.html
  Saved: figS_shap_HRC.svg | figS_shap_HRC.png | figS_shap_HRC.html


### Tables, pickled models, raw SHAP arrays

* `tables/perf_summary.csv` collects train/test R^2 and MAE for the three estimators (CART, RF, GBT) at every tier.
* `tables/shap_per_feature_<tier>.csv` is the per-feature SHAP ranking used for the cross-tier panel.
* `tables/headline_matrix.parquet` is the design matrix used to fit the models, kept for reproducibility.
* `models/<tier>_models.pkl` holds the fitted CART, RF, and GBT estimators for one tier.
* `models/<tier>_shap.npz` holds the raw SHAP values, the SHAP-sample feature matrix, the column names, and the region of every SHAP-sample row.
* `_config.json` records the run flags (feature year, target year, PR level, exemplars, features, random state, tune flag).

In [14]:
perf_rows = []
for tier, fit in fits.items():
    for m, met in fit['metrics'].items():
        if m == 'n_total':
            continue
        row = {
            'stratum': tier,
            'n_total': fit['metrics']['n_total'],
            'model':   m,
            **met,
        }
        row.pop('best_params', None)
        perf_rows.append(row)
perf_df = pd.DataFrame(perf_rows)
perf_df.to_csv(FIG_DIR / 'tables' / 'perf_summary.csv', index=False)

for tier in RC_ORDER:
    sf = shap_per_feature(shaps[tier], feat_cols)
    sf.to_csv(
        FIG_DIR / 'tables' / f'shap_per_feature_{tier}.csv', index=False,
    )

matrix.to_parquet(FIG_DIR / 'tables' / 'headline_matrix.parquet')

for tier, fit in fits.items():
    with open(FIG_DIR / 'models' / f'{tier}_models.pkl', 'wb') as f:
        pickle.dump({
            'cart': fit['cart'],
            'rf':   fit['rf'],
            'gbt':  fit['gbt'],
        }, f)
    np.savez(
        FIG_DIR / 'models' / f'{tier}_shap.npz',
        shap_values=shaps[tier]['shap_values'],
        X_shap=shaps[tier]['X_shap'].values,
        X_columns=np.array(feat_cols),
        regions=shaps[tier]['regions_shap'].values,
    )

cfg = {
    'merged_dir':   str(MERGED_DIR),
    'headline':     HEADLINE,
    'target':       TARGET,
    'exemplars':    EXEMPLARS,
    'features':     feat_cols,
    'random_state': RANDOM_STATE,
    'tune':         TUNE,
}
(FIG_DIR / '_config.json').write_text(json.dumps(cfg, indent=2))

print('Run folder:', FIG_DIR.resolve())
print('\nperf_summary.csv:')
perf_df.round(3)

Run folder: C:\Users\Sarfr001\Documents\Global_Potential_Potable_Reuse\outputs\figure4

perf_summary.csv:


,stratum,n_total,model,train_r2,test_r2,test_mae
0,LRC,1632,cart,0.484,0.468,3.710
1,LRC,1632,rf,0.726,0.700,2.629
2,LRC,1632,gbt,0.998,0.996,0.284
3,MRC,1632,cart,0.618,0.566,5.501
4,MRC,1632,rf,0.883,0.874,2.518
5,MRC,1632,gbt,1.000,0.999,0.214
6,HRC,1632,cart,0.413,0.334,1.563
7,HRC,1632,rf,0.802,0.764,0.636
8,HRC,1632,gbt,1.000,0.999,0.046


### Inspect outputs

In [15]:
print('=== SHAP rankings per tier ===')
for tier in RC_ORDER:
    df = pd.read_csv(
        FIG_DIR / 'tables' / f'shap_per_feature_{tier}.csv'
    )
    print(f'\n{tier}:')
    print(df[['pretty', 'mean_abs', 'mean_signed']]
          .round(3).to_string(index=False))

print('\nMain figure files:')
for f in sorted((FIG_DIR / 'figures').glob('figure4_main.*')):
    print(' ', f.resolve())

print('\nSupplementary HTML:')
for f in sorted((FIG_DIR / 'supplementary').glob('*.html')):
    print(' ', f.name)

=== SHAP rankings per tier ===

LRC:
                pretty  mean_abs  mean_signed
        GDP per capita     2.964       -0.431
Groundwater withdrawal     2.072        0.240
            Population     1.593        0.131
  Municipal withdrawal     1.573        0.060

MRC:
                pretty  mean_abs  mean_signed
Groundwater withdrawal     6.288        0.165
            Population     3.221        0.154
        GDP per capita     2.737       -0.369
  Municipal withdrawal     2.697        0.049

HRC:
                pretty  mean_abs  mean_signed
Groundwater withdrawal     1.184       -0.116
        GDP per capita     0.845       -0.014
            Population     0.593        0.076
  Municipal withdrawal     0.580        0.054

Main figure files:
  C:\Users\Sarfr001\Documents\Global_Potential_Potable_Reuse\outputs\figure4\figures\figure4_main.html
  C:\Users\Sarfr001\Documents\Global_Potential_Potable_Reuse\outputs\figure4\figures\figure4_main.png
  C:\Users\Sarfr001\Documents\Global

## Vertical SHAP beeswarm


In [16]:
from matplotlib import colormaps


SHAP_VALUE_CMAP = colormaps['cividis']


def draw_shap_vertical(
    ax, shap_pkg, feat_cols,
    panel_label='', panel_subtitle='SHAP attribution, medium reuse cost',
    jitter=0.20, point_size=10,
):
    sv = shap_pkg['shap_values']
    X = shap_pkg['X_shap']
    pretty_feats = [pretty_name(c) for c in feat_cols]

    # Column order: strongest mean |SHAP| first.
    order_idx = np.argsort(-np.abs(sv).mean(axis=0))
    n_feats = len(feat_cols)
    x_positions = np.arange(n_feats)
    rng = np.random.default_rng(RANDOM_STATE)

    for plot_col, j in enumerate(order_idx):
        s = sv[:, j]
        x_vals = X.iloc[:, j].values.astype(float)

        lo, hi = np.nanmin(x_vals), np.nanmax(x_vals)
        if hi > lo:
            v_norm = (x_vals - lo) / (hi - lo)
        else:
            v_norm = np.full_like(x_vals, 0.5)

        x_jitter = rng.uniform(-jitter, jitter, size=len(s))
        x_col = np.full_like(s, x_positions[plot_col], dtype=float) + x_jitter

        ax.scatter(
            x_col, s, c=v_norm, cmap=SHAP_VALUE_CMAP,
            s=point_size, alpha=0.65, edgecolor='none',
            vmin=0, vmax=1,
        )


    ax.axhline(0, color='#888', linewidth=0.6, zorder=0, linestyle='--')

    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [pretty_feats[j] for j in order_idx],
        rotation=45, fontsize=14, ha='right', rotation_mode='anchor',
    )
    ax.set_ylabel('Shapley value (percentage points)', fontsize=14)
    ax.tick_params(axis='y', labelsize=12)
    ax.set_xlim(-0.5, n_feats - 0.5)

    # Horizontal colorbar above the axes. Title above the bar, Low/High
    # tick labels below it, Low on the left.
    cax = ax.inset_axes([0.10, 1.05, 0.80, 0.025])
    sm = plt.cm.ScalarMappable(cmap=SHAP_VALUE_CMAP, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cb = plt.colorbar(sm, cax=cax, orientation='horizontal')
    cb.set_ticks([0.05, 0.95])
    cb.set_ticklabels(['Low', 'High'])
    cb.ax.tick_params(labelsize=11, length=2)
    cb.outline.set_linewidth(0.4)
    cax.set_title('Attribute value', fontsize=12, color='#444', pad=4)


    panel_y = 1.20
    if panel_label:
        ax.text(
            -0.18, panel_y, panel_label,
            fontsize=16, fontweight='bold',
            ha='left', va='bottom', transform=ax.transAxes,
        )
    if panel_subtitle:
        x_sub = -0.10 if panel_label else -0.18
        ax.text(
            x_sub, panel_y, panel_subtitle,
            fontsize=14, color='#444',
            ha='left', va='bottom', transform=ax.transAxes,
        )

fig, ax = plt.subplots(figsize=(8, 9.0))
fig.subplots_adjust(left=0.16, right=0.95, top=0.82, bottom=0.22)

draw_shap_vertical(
    ax, shaps[TIER], feat_cols,
    panel_subtitle='SHAP attribution, medium reuse cost',
)

save_fig(fig, FIG_DIR / 'figures', 'figure4_shap_vertical')
print('Wrote vertical SHAP figure to:',
      (FIG_DIR / 'figures' / 'figure4_shap_vertical.svg').resolve())


  Saved: figure4_shap_vertical.svg | figure4_shap_vertical.png | figure4_shap_vertical.html
Wrote vertical SHAP figure to: C:\Users\Sarfr001\Documents\Global_Potential_Potable_Reuse\outputs\figure4\figures\figure4_shap_vertical.svg
